# Re-ingest 15 luật (có nhãn Điều) + so sánh RAG vs Graph-RAG — chạy trên Colab GPU

**Bối cảnh:** vectorstore cũ thiếu metadata `article` → cầu nối KG→chunk chết → Graph RAG = RAG.
Notebook này nạp lại 15 luật bằng pipeline hiện tại (gắn `article` đầy đủ) trên **GPU**, rồi đo lại
để thấy Graph RAG bắt đầu đóng góp chunk riêng (`only_GR > 0`).

**Trước khi chạy:** menu `Runtime → Change runtime type → T4 GPU`.

Các bước: ① kiểm tra GPU → ② upload `colab_bundle.zip` → ③ cài deps → ④ nhập Neo4j → ⑤ chạy → ⑥ tải kết quả về.

## ① Kiểm tra GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'CHƯA bật GPU! Runtime → Change runtime type → T4 GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))

## ② Upload `colab_bundle.zip` và giải nén

Bundle (1.7 MB) gồm: `src/`, `scripts/`, `requirements.txt`, manifest 15 luật và 14 file luật thô.
Chạy cell, bấm **Choose Files**, chọn `colab_bundle.zip` trên máy bạn.

In [ ]:
import os, zipfile
from google.colab import files
up = files.upload()  # chọn colab_bundle.zip
name = next(iter(up))
os.makedirs('/content/proj', exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall('/content/proj')
os.chdir('/content/proj')
print('CWD:', os.getcwd())
print('Có:', sorted(os.listdir('.')))
print('Raw luật:', len(os.listdir('data/raw/all_laws')), 'file')

## ③ Cài thư viện

Chỉ cài đúng các gói pipeline cần (nhanh hơn cài cả `requirements.txt`). `sentence-transformers` kéo theo `torch` đã có sẵn trên Colab.

In [ ]:
!pip -q install chromadb==1.5.9 sentence-transformers neo4j rank-bm25 langchain-text-splitters loguru pydantic pydantic-settings python-dotenv
print('Done.')

## ④ Cấu hình Neo4j + đường dẫn

Dán thông tin Neo4j Aura **từ file `.env` của bạn** vào dưới (KHÔNG commit/đưa secret cho ai). KG dùng để map Điều → chunk khi so sánh.

In [ ]:
import os
# >>> DÁN GIÁ TRỊ TỪ .env CỦA BẠN <<<
os.environ['NEO4J_URI']      = 'neo4j+s://XXXX.databases.neo4j.io'
os.environ['NEO4J_USERNAME'] = 'XXXX'
os.environ['NEO4J_PASSWORD'] = 'XXXX'
os.environ['NEO4J_DATABASE'] = 'XXXX'

os.environ['EMBEDDING_MODEL'] = 'BAAI/bge-m3'
os.environ['VECTOR_BACKEND']  = 'chroma'
# Không gọi LLM trong notebook này — đặt provider giả để tránh lỗi import
os.environ.setdefault('LLM_PROVIDER', 'ollama')
print('Đã set env. Neo4j URI:', os.environ['NEO4J_URI'])

## ⑤ Chạy re-ingest (GPU) + so sánh

Nạp 14 luật → ~18k chunk, embed bằng bge-m3 trên GPU (fp16). Trên T4 mất khoảng **5–15 phút**.
Cuối ra bảng `only_GR` (số chunk Graph-RAG có mà RAG không có) và `KG_map` (số hit KG map được ra chunk).

In [ ]:
!python -m scripts.rebuild_top15_and_compare

## ⑥ Đóng gói kết quả để tải về máy

Nén `data/vectorstore` (collection `legal_top15`) + BM25 index. Tải về rồi giải nén đè vào thư mục `data/` của project local.

In [ ]:
import shutil, os
os.makedirs('/content/out/data/bm25', exist_ok=True)
shutil.copytree('data/vectorstore', '/content/out/data/vectorstore', dirs_exist_ok=True)
shutil.copy('data/bm25/top15_index.json', '/content/out/data/bm25/top15_index.json')
shutil.make_archive('/content/top15_store', 'zip', '/content/out')
print('Size:', round(os.path.getsize('/content/top15_store.zip')/1e6, 1), 'MB')
from google.colab import files
files.download('/content/top15_store.zip')

## Dùng lại ở máy local (KHÔNG cần embed lại)

1. Giải nén `top15_store.zip` → được `data/vectorstore/` và `data/bm25/top15_index.json`.
2. Chép đè vào `ProjectGenAI_2/data/` ở máy bạn (gộp vào `data/vectorstore` và `data/bm25` hiện có — collection `legal_top15` là tên riêng nên không đụng `legal_docs` cũ).
3. Chạy so sánh local, không phải embed lại:
   ```bash
   python -m scripts.rebuild_top15_and_compare --compare-only
   ```
   Cột **only_GR** > 0 ở các câu KG bắt được Điều = Graph RAG đã vượt RAG.

> Muốn nạp **toàn corpus** (không chỉ 15 luật): upload cả `data/raw/` đầy đủ rồi chạy `python -m scripts.ingest --reset` và `python -m scripts.build_bm25` thay cho bước ⑤ — nhưng dữ liệu raw ~2GB, cân nhắc dùng Google Drive thay vì upload trực tiếp.